# Model training and evaluation

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [19]:
train_df = pd.read_csv("../data/train_data.csv")
val_df = pd.read_csv("../data/val_data.csv")
test_df = pd.read_csv("../data/test_data.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (1273376, 21)
Val: (173886, 21)
Test: (173887, 21)


In [20]:
label_col = "Label"

X_train = train_df.drop(columns=[label_col])
y_train = train_df[label_col]

X_val = val_df.drop(columns=[label_col])
y_val = val_df[label_col]

X_test = test_df.drop(columns=[label_col])
y_test = test_df[label_col]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (1273376, 20) y_train: (1273376,)
X_val: (173886, 20) y_val: (173886,)
X_test: (173887, 20) y_test: (173887,)


In [21]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    )

model.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

In [ ]:
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    136433
           1       1.00      1.00      1.00     37453

    accuracy                           1.00    173886
   macro avg       1.00      1.00      1.00    173886
weighted avg       1.00      1.00      1.00    173886



In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn + 1e-9)
print("FPR:", fpr)

[[136433      0]
 [    14  37439]]
FPR: 0.0


In [ ]:
thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    y_pred_t = (y_val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_t).ravel()

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)

    print(f"T={t:.1f} | Precision={precision:.3f} | Recall={recall:.3f} | FPR={fpr:.3f}")

BEST_THRESHOLD = 0.6  # update after reviewing the table

T=0.1 | Precision=0.996 | Recall=1.000 | FPR=0.001
T=0.2 | Precision=0.999 | Recall=1.000 | FPR=0.000
T=0.3 | Precision=1.000 | Recall=1.000 | FPR=0.000
T=0.4 | Precision=1.000 | Recall=1.000 | FPR=0.000
T=0.5 | Precision=1.000 | Recall=1.000 | FPR=0.000
T=0.6 | Precision=1.000 | Recall=0.999 | FPR=0.000
T=0.7 | Precision=1.000 | Recall=0.999 | FPR=0.000
T=0.8 | Precision=1.000 | Recall=0.999 | FPR=0.000


In [ ]:
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= BEST_THRESHOLD).astype(int)

print(classification_report(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print("FPR:", fp / (fp + tn + 1e-9))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    136434
           1       1.00      1.00      1.00     37453

    accuracy                           1.00    173887
   macro avg       1.00      1.00      1.00    173887
weighted avg       1.00      1.00      1.00    173887

[[136434      0]
 [    13  37440]]
FPR: 0.0


In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importance)

PacketLengthMode                      0.305612
Duration                              0.122948
PacketLengthMedian                    0.083096
PacketLengthMean                      0.079150
FlowBytesReceived                     0.063791
FlowBytesSent                         0.059488
ResponseTimeTimeMedian                0.050922
PacketLengthVariance                  0.038299
PacketTimeMedian                      0.029615
PacketLengthSkewFromMode              0.028056
PacketLengthSkewFromMedian            0.025165
FlowReceivedRate                      0.023579
ResponseTimeTimeSkewFromMode          0.019439
ResponseTimeTimeSkewFromMedian        0.016956
PacketLengthCoefficientofVariation    0.014941
ResponseTimeTimeMean                  0.010494
FlowSentRate                          0.009690
PacketTimeCoefficientofVariation      0.006507
PacketTimeSkewFromMode                0.006227
PacketTimeSkewFromMedian              0.006026
dtype: float64


In [ ]:
# Stress testing slices
low_rate = X_test[X_test["FlowSentRate"] < X_test["FlowSentRate"].quantile(0.25)]
low_rate_pred = model.predict(low_rate)
print("Low-rate malicious ratio:", low_rate_pred.mean())

cdn_like = X_test[X_test["FlowBytesReceived"] > X_test["FlowBytesReceived"].quantile(0.75)]
cdn_pred = model.predict(cdn_like)
print("CDN false positive ratio:", cdn_pred.mean())

short_flows = X_test[X_test["Duration"] < X_test["Duration"].quantile(0.1)]
long_flows = X_test[X_test["Duration"] > X_test["Duration"].quantile(0.9)]
print("Short flow predictions:", model.predict(short_flows).mean())
print("Long flow predictions:", model.predict(long_flows).mean())

Low-rate malicious ratio: 0.4118743099006257
CDN false positive ratio: 0.30585204269414795
Short flow predictions: 0.001725327812284334
Long flow predictions: 0.7256728778467909


**The model achieved near-perfect metrics, but stress testing revealed that it relies heavily on flow size and duration. It struggles with low-rate tunneling and produces high false positives on CDN-like traffic, indicating that the dataset lacks realistic overlap between benign and malicious behavior**